In [1]:
import pandas as pd

# Load dataset 
df = pd.read_csv("Matuu_hospital.csv", on_bad_lines="skip")
print(df.head())

# Checking original Columns
print(df.columns) 
df1b = df.drop(
    columns="Unnamed: 0",
    errors="ignore"
)

# Drop duplicates, completely empty rows, and completely empty columns
df1 = (df1b.drop_duplicates()
      .dropna(axis=0, how="all")
      .dropna(axis=1, how="all")
      .reset_index(drop=True)
     )

# Clean column names
df1.columns = (df1.columns
    .str.replace(r"[^a-zA-Z0-9]", " ", regex=True)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
)

# Rename columns
column_updates = {
    "hasinsurance_verify": "has_insurance_verify", 
    "dischargedate": "discharge_date",
    "admissiondate": "admission_date",
    "medicalcondition": "medical_condition", 
    "patientname": "patient_name", 
    "patientid": "patient_id", 
}

df1 = df1.rename(columns=column_updates)

# Sort df by patient ID 
df1 = df1.sort_values(
    by="patient_id", 
    ascending=True
)

# Checking duplicated patient ids
print(df1["patient_name"].duplicated().sum())
print(df1["patient_name"].isna().sum())

# Checking duplicated patient ids
print(df1.loc[df1["patient_id"].duplicated(keep=False)])

# Clean duplicated ids, using keys from patient name column
update_patient_id = {
    "Patient_11": "P111_A", 
    "Patient_20": "P120_A", 
}

df1["patient_id"] = (
    df1["patient_name"].map(update_patient_id)
    .fillna(df1["patient_id"])
)
print(df1.loc[df1["patient_id"].duplicated(keep=False)]) # No duplicates

print(df1["patient_id"].nunique())
print(df1["patient_id"].isna().sum()) # No NaN

# Cleaning medical_condition column
print(df1["medical_condition"].value_counts())

df1["medical_condition"] = (df1["medical_condition"]
                             .str.strip()
                             .str.lower()
                            )

med_updates = {
    "typhoid": "Typhoid", 
    "malaria": "Malaria", 
    "diabetes": "Diabetes", 
    "tb": "TB", 
    "cholera": "Cholera", 
    "hypertension": "Hypertension", 
    "covid-19": "Covid-19", 
    "malariaa": "Malaria", 
    "diabtes": "Diabetes", 
}

df1["medical_condition_clean"] = df1["medical_condition"].map(med_updates)
print(df1["medical_condition_clean"].isna().sum())

df1 = df1.dropna(subset=["medical_condition_clean"])

# Cleaning has_insurance_verify column
print(df1["has_insurance_verify"].value_counts())

update_has_insurance_verify = {
    0: "No", 
    "Yes": "Yes", 
    "TRUE": "Yes", 
    "FALSE": "No", 
    "No": "No", 
    "N": "No", 
    "Y": "Yes", 
    "n": "No", 
    "y": "Yes", 
    "yes": "Yes", 
    1: "Yes", 
    "False": "No", 
    "no": "No", 
    "True": "Yes", 
    "0": "No", 
    "1": "Yes", 
}

df1["has_insurance_verify_clean"] = pd.Categorical(
    df1["has_insurance_verify"].map(update_has_insurance_verify), 
    categories=["Yes", "No"], 
    ordered=False, 
)
print(df1["has_insurance_verify_clean"].value_counts())

# Cleaning "age" column
df1["age_clean"] = pd.to_numeric(
    df1["age"], 
    errors="coerce", 
)

broken_ages = df1.loc[(df1["age_clean"].isna()) & (df1["age"].notna())]
print(broken_ages)

df1["age"] = df1["age"].astype(str).str.replace("I", "1")

df1["age_clean"] = pd.to_numeric(
    df1["age"], 
    errors="coerce", 
)

broken_ages_2 = df1.loc[(df1["age_clean"].isna()) & (df1["age"].notna())]
print(broken_ages_2)

# No broken ages, the column is now clean
print(df1["age_clean"].isna().sum())
df1 = df1.dropna(subset=["age_clean"])
print(df1["age_clean"].isna().sum()) # Confirmed, no NaN cells in age

print(df1["admission_date"].head()) # To know the date formats

df1["admission_date_clean"] = pd.to_datetime(
    df1["admission_date"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)

# Checking invalid admission dates
invalid_adm_dates = df1.loc[(df1["admission_date"].notna()) & (df1["admission_date_clean"].isna())]
print(invalid_adm_dates) # Some dates have O instead of 0, other I instead of 1

df1["admission_date_text"] = (df1["admission_date"]
                              .astype(str)
                              .str.replace("O", "0")
                              .str.replace("I", "1")
                             )
df1["admission_date_clean"] = pd.to_datetime(
    df1["admission_date_text"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)
invalid_adm_dates_2 = df1.loc[(df1["admission_date_text"].notna()) & (df1["admission_date_clean"].isna())]
print(invalid_adm_dates_2) # Cleaned, invalid dates now

print(df1["discharge_date"].head()) # To know discharge dates format

df1["discharge_date_clean"] = pd.to_datetime(
    df1["discharge_date"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)

invalid_discharge_dates = df1.loc[(df1["discharge_date_clean"].isna()) & (df1["discharge_date"].notna())]
print(invalid_discharge_dates) # There are invalid dates with O instead of 0

df1["discharge_date_text"] = df1["discharge_date"].astype(str).str.replace("O", "0")

df1["discharge_date_clean"] = pd.to_datetime(
    df1["discharge_date_text"], 
    format="%d/%m/%Y", 
    errors="coerce", 
)

# Confirming that there are no invalid dates
invalid_discharge_dates_2 = df1.loc[(df1["discharge_date_clean"].isna()) & (df1["discharge_date_text"].notna())]
print(invalid_discharge_dates_2) # No invalid dates

print(df1["discharge_date_clean"].isna().sum())

# Checking broken dates
broken_dates = df1["discharge_date_clean"] < df1["admission_date_clean"]
show_rows_broken_dates = df1.loc[broken_dates]
print(show_rows_broken_dates) # No broken dates

# df has broken dates, let's clean them (Update path to your local environment file)
df2 = pd.read_csv("broken_dates.csv", on_bad_lines="skip")
print(df2.info())

print(df2["Admission_Date"].head())

df2["Admission_Date_clean"] = pd.to_datetime(
    df2["Admission_Date"], 
    format="mixed", 
    errors="coerce", 
)

print(df2["Admission_Date_clean"].isna().sum())
print(df2.loc[(df2["Admission_Date_clean"].isna()) & (df2["Admission_Date"].notna())])
print(df2["Admission_Date_clean"].head())

admission_dates_updates = {
    "PID-4432": pd.Timestamp("2023-11-30"), 
    "PID-3321": pd.Timestamp("2023-02-28"), 
}

df2["Admission_Date_clean"] = (df2["Patient_ID"].map(admission_dates_updates)
                               .fillna(df2["Admission_Date_clean"])
                              )

print(df2["Admission_Date_clean"].isna().sum())
print(df2["Admission_Date"].isna().sum())

df2 = df2.dropna(subset="Admission_Date_clean")
print(df2["Admission_Date_clean"].isna().sum())

df2["Discharge_Date_clean"] = pd.to_datetime(
    df2["Discharge_Date"], 
    format="mixed", 
    errors="coerce", 
)

print(df2.loc[(df2["Discharge_Date_clean"].isna()) & (df2["Discharge_Date"].notna())])
print(df2["Discharge_Date_clean"].dtype)

brokenn_dates = df2["Discharge_Date_clean"] < df2["Admission_Date_clean"]
print(df2.loc[brokenn_dates])

df2.loc[brokenn_dates, ["Discharge_Date_clean", "Admission_Date_clean"]] = df2.loc[brokenn_dates, ["Admission_Date_clean", "Discharge_Date_clean"]].values

brokenn_dates_2 = df2["Discharge_Date_clean"] < df2["Admission_Date_clean"]
print(df2.loc[brokenn_dates_2]) # Clean, no broken dates


   Unnamed: 0 PatientID#   PatientName   Age MedicalCondition AdmissionDate  \
0           0       P100      Patient_0  NaN                            NaN   
1           1       P101      Patient_1   29          Typhoid    04/06/2026   
2           2       P102      Patient_2   70          Malaria    14/05/2026   
3           3       P103      Patient_3  12I          Typhoid    17/02/2026   
4           4       P104      Patient_4   72          Typhoid    21/06/2026   

  DischargeDate HasInsurance_Verify  
0    29/01/2026               False  
1    15/06/2026                   N  
2    15/05/2026                 Yes  
3    27/02/2026                 Yes  
4    03/07/2026                TRUE  
Index(['Unnamed: 0', 'PatientID#', '  PatientName ', 'Age', 'MedicalCondition',
       'AdmissionDate', 'DischargeDate', 'HasInsurance_Verify'],
      dtype='object')
Index(['PatientID#', '  PatientName ', 'Age', 'MedicalCondition',
       'AdmissionDate', 'DischargeDate', 'HasInsurance_Verify'],

In [2]:
# Working with dates
df1["length_of_stay_days"]= df1["discharge_date_clean"]- df1["admission_date_clean"]
print("df created successfully")
df1["length_of_stay"]=df1["length_of_stay_days"]/pd.to_timedelta(1, unit= "D")
print(df1["length_of_stay"].head())

df1["Onset_date"]= df1["admission_date_clean"]-pd.Timedelta(days= 3)
print(df1["Onset_date"].head())

df1["discharge_date_clean_copy"]= df1["admission_date_clean"] + (pd.to_timedelta(
    df1["length_of_stay"], 
    unit = "D")
                                                                )

print(df1["discharge_date_clean_copy"].head())

df1["length_of_stay_math_numbers"]= df1["length_of_stay_days"].dt.days
print(df1["length_of_stay_math_numbers"].head())

df1["admission_day_names"] = df1["admission_date_clean"].dt.day_name()
print(df1["admission_day_names"].head())
df1["admission_month_name"] = df1["admission_date_clean"].dt.month_name()
print(df1["admission_month_name"].head())

df1["admission_month"] = df1["admission_date_clean"].dt.month
print(df1["admission_month"].head())

df1["admission_day_of_month"] = df1["admission_date_clean"].dt.day
print(df1["admission_day_of_month"].head())
df1["admission_day_of_week"] = df1["admission_date_clean"].dt.dayofweek
print(df1["admission_day_of_week"].head())

df1["admission_year"] = df1["admission_date_clean"].dt.year
df1["year_after_admission_date"]= df1["admission_date_clean"] + pd.DateOffset(years = 1)
print(df1["year_after_admission_date"].dt.year.head())
print("==="*25)
print(df1["admission_year"].head())
df1["a_week_after_admission"]= df1["admission_date_clean"] + pd.Timedelta(days= 7)
df1["a_week_after_admission_m2"]= df1["admission_date_clean"] + pd.Timedelta(weeks= 1)
df1["a_month_after_admission_date"]= df1["admission_date_clean"] + pd.DateOffset(months=1)
df1["name_of_month"]= df1["a_month_after_admission_date"].dt.month_name()
print(df1["name_of_month"].head())
df1["epi_week"]= df1["admission_date_clean"].dt.isocalendar().week
print(df1["epi_week"].head())







<class 'pandas.core.frame.DataFrame'>
Index: 91 entries, 1 to 99
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype          
---  ------                        --------------  -----          
 0   patient_id                    91 non-null     object         
 1   patient_name                  91 non-null     object         
 2   age                           91 non-null     object         
 3   medical_condition             91 non-null     object         
 4   admission_date                87 non-null     object         
 5   discharge_date                91 non-null     object         
 6   has_insurance_verify          91 non-null     object         
 7   medical_condition_clean       91 non-null     category       
 8   has_insurance_verify_clean    91 non-null     category       
 9   age_clean                     91 non-null     float64        
 10  admission_date_clean          87 non-null     datetime64[ns] 
 11  admission_date_text       

In [3]:
print(df1["admission_date_clean"].head())

1   2026-06-04
2   2026-05-14
3   2026-02-17
4   2026-06-21
5   2026-04-19
Name: admission_date_clean, dtype: datetime64[ns]


In [4]:
snap_date = pd.to_datetime("2026-05-15")
hos_occupancy_df= df1.loc[
    (df1["admission_date_clean"]<=snap_date)&
    (
        (df1["discharge_date_clean"]> snap_date)|
    (df1["discharge_date_clean"].isna())
    )
    ].copy()

print(hos_occupancy_df.shape[0])

hos_occupancy_df["los"]= snap_date-hos_occupancy_df["admission_date_clean"]

print(hos_occupancy_df["los"].head())

hos_occupancy_df["los_raw_numbers"]= hos_occupancy_df["los"].dt.days
print(hos_occupancy_df["los_raw_numbers"].mean())

print(hos_occupancy_df["patient_id"].unique())


5
43    0 days
48   10 days
55    5 days
66    0 days
98    2 days
Name: los, dtype: timedelta64[ns]
3.4
['P143' 'P148' 'P155' 'P166' 'P198']


In [5]:
print(df1["patient_id"].head(10).unique())
print(df1["patient_id"].nunique())
print(df1["patient_id"].count())
print(df1["patient_id"].size)
print(df1["patient_id"].shape[0])

['P101' 'P102' 'P103' 'P104' 'P105' 'P106' 'P107' 'P108' 'P109' 'P110']
91
91
91
91


In [6]:
df3= pd.read_csv(r"D:\Public health\epidemic_data1.csv", parse_dates= ["AdmissionDate", "DischargeDate"], on_bad_lines="skip")
print(df3.head())
print(df3.info())
df3.columns= df3.columns.str.strip()


  PatientID    PatientName  Age MedicalCondition AdmissionDate DischargeDate  \
0      P001       John Doe   45        Influenza    2026-05-01    2026-05-05   
1      P002     Jane Smith   29         COVID-19    2026-05-02    2026-05-18   
2      P003    Bob Johnson   67        Influenza    2026-05-04    2026-05-04   
3      P004    Alice Brown   82         COVID-19    2026-05-04           NaT   
4      P005  Charlie Green   12  Gastroenteritis    2026-05-04    2026-05-06   

  HasInsurance_Verify  
0                 Yes  
1                 Yes  
2                  No  
3                 Yes  
4                 Yes  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   PatientID            20 non-null     object        
 1   PatientName          20 non-null     object        
 2   Age                  20 non-null     int

In [7]:
my_serie= df3.groupby("AdmissionDate").size()
print(my_serie.index)
print(my_serie.values)
print(my_serie)
#This will show the number of patients admitted in each date
my_new_df= my_serie.reset_index(name="raw_cases")
print(my_new_df)

#Method 2, just for practising
new_2_df= df3.groupby("AdmissionDate").agg(
    raw_cases = ("AdmissionDate", "size")
).reset_index()

print(new_2_df)
#I think this step is unnecessary
my_new_df= my_new_df.sort_values(
    "AdmissionDate"
).reset_index(drop=True)

print(my_new_df)

# I have seen that, each and everytime I run .sort_values(), it's good to chain .reset_index(drop=True)
busiest_days_df = my_new_df.sort_values(
    by="raw_cases", 
    ascending=False, 
).reset_index(drop=True)

print(busiest_days_df )
#Find daily average
print(my_new_df["raw_cases"].mean())
#Find total weekly cases

# Find out the day with many patients/cases

my_new_df["day_of_week"]= my_new_df["AdmissionDate"].dt.day_name()

print(my_new_df["day_of_week"].head())

day_total_cases= my_new_df.groupby("day_of_week")["raw_cases"].sum()

max_day_total_cases= day_total_cases.max()

days_with_max_cases= day_total_cases[day_total_cases==max_day_total_cases]
    
print(days_with_max_cases.index)

series_without_max_day= day_total_cases[day_total_cases!=max_day_total_cases]

day_with_second_highest_case= series_without_max_day.max()

days_with_2nd_max_cases= series_without_max_day[series_without_max_day==day_with_second_highest_case]
print(days_with_2nd_max_cases.index)

print(day_total_cases)

day_wit_max_cases= my_new_df.groupby("day_of_week")["raw_cases"].transform("max")

print(my_new_df.loc[my_new_df["raw_cases"]==day_wit_max_cases])


     

DatetimeIndex(['2026-05-01', '2026-05-02', '2026-05-04', '2026-05-08',
               '2026-05-11', '2026-05-13', '2026-05-15', '2026-05-18',
               '2026-05-19'],
              dtype='datetime64[ns]', name='AdmissionDate', freq=None)
[1 1 4 1 5 1 2 4 1]
AdmissionDate
2026-05-01    1
2026-05-02    1
2026-05-04    4
2026-05-08    1
2026-05-11    5
2026-05-13    1
2026-05-15    2
2026-05-18    4
2026-05-19    1
dtype: int64
  AdmissionDate  raw_cases
0    2026-05-01          1
1    2026-05-02          1
2    2026-05-04          4
3    2026-05-08          1
4    2026-05-11          5
5    2026-05-13          1
6    2026-05-15          2
7    2026-05-18          4
8    2026-05-19          1
  AdmissionDate  raw_cases
0    2026-05-01          1
1    2026-05-02          1
2    2026-05-04          4
3    2026-05-08          1
4    2026-05-11          5
5    2026-05-13          1
6    2026-05-15          2
7    2026-05-18          4
8    2026-05-19          1
  AdmissionDate  raw_cases

In [8]:
print(df3.groupby("AdmissionDate")["PatientID"].count())
print(df3.groupby("AdmissionDate")["PatientID"].size())
print(df3.groupby("AdmissionDate")["PatientID"].nunique())

AdmissionDate
2026-05-01    1
2026-05-02    1
2026-05-04    4
2026-05-08    1
2026-05-11    5
2026-05-13    1
2026-05-15    2
2026-05-18    4
2026-05-19    1
Name: PatientID, dtype: int64
AdmissionDate
2026-05-01    1
2026-05-02    1
2026-05-04    4
2026-05-08    1
2026-05-11    5
2026-05-13    1
2026-05-15    2
2026-05-18    4
2026-05-19    1
Name: PatientID, dtype: int64
AdmissionDate
2026-05-01    1
2026-05-02    1
2026-05-04    4
2026-05-08    1
2026-05-11    5
2026-05-13    1
2026-05-15    2
2026-05-18    4
2026-05-19    1
Name: PatientID, dtype: int64


DatetimeIndex(['2026-05-01', '2026-05-02', '2026-05-04', '2026-05-08',
               '2026-05-11', '2026-05-13', '2026-05-15', '2026-05-18',
               '2026-05-19'],
              dtype='datetime64[ns]', name='AdmissionDate', freq=None)
[1 1 4 1 5 1 2 4 1]
AdmissionDate
2026-05-01    1
2026-05-02    1
2026-05-04    4
2026-05-08    1
2026-05-11    5
2026-05-13    1
2026-05-15    2
2026-05-18    4
2026-05-19    1
dtype: int64
  AdmissionDate  raw_cases
0    2026-05-01          1
1    2026-05-02          1
2    2026-05-04          4
3    2026-05-08          1
4    2026-05-11          5
5    2026-05-13          1
6    2026-05-15          2
7    2026-05-18          4
8    2026-05-19          1


In [32]:
my_serie= df3.groupby("AdmissionDate").size()
print(my_serie.index)
print(my_serie.values)
print(my_serie)
#This will show the number of patients admitted in each date
my_new_df= my_serie.reset_index(name="raw_cases")
print(my_new_df)

my_new_df["day_of_week"]= my_new_df["AdmissionDate"].dt.day_name()

print(my_new_df.head())

ts_df= my_new_df.set_index(
    "AdmissionDate"
)

ts_df["true_7_day_avg"] = ts_df["raw_cases"].rolling(
    window="7D"
).mean()

print(ts_df["true_7_day_avg"].head())

print(df3.info())
print(my_new_df)

path_out= (r"D:\Public health\my_new_df.csv")
my_new_df.to_csv(path_out, index=False, encoding= "utf-8-sig")
print("File written successfully")


DatetimeIndex(['2026-05-01', '2026-05-02', '2026-05-04', '2026-05-08',
               '2026-05-11', '2026-05-13', '2026-05-15', '2026-05-18',
               '2026-05-19'],
              dtype='datetime64[ns]', name='AdmissionDate', freq=None)
[1 1 4 1 5 1 2 4 1]
AdmissionDate
2026-05-01    1
2026-05-02    1
2026-05-04    4
2026-05-08    1
2026-05-11    5
2026-05-13    1
2026-05-15    2
2026-05-18    4
2026-05-19    1
dtype: int64
  AdmissionDate  raw_cases
0    2026-05-01          1
1    2026-05-02          1
2    2026-05-04          4
3    2026-05-08          1
4    2026-05-11          5
5    2026-05-13          1
6    2026-05-15          2
7    2026-05-18          4
8    2026-05-19          1
  AdmissionDate  raw_cases day_of_week
0    2026-05-01          1      Friday
1    2026-05-02          1    Saturday
2    2026-05-04          4      Monday
3    2026-05-08          1      Friday
4    2026-05-11          5      Monday
AdmissionDate
2026-05-01    1.0
2026-05-02    1.0
2026-05-04  

In [34]:
print(df1.info())

df1_path_out= (r"D:\Public health\4th_sep_2026.csv")
df1.to_csv(df1_path_out, index=False, encoding= "utf-8-sig")
print("File written successfully")

<class 'pandas.core.frame.DataFrame'>
Index: 91 entries, 1 to 99
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype          
---  ------                        --------------  -----          
 0   patient_id                    91 non-null     object         
 1   patient_name                  91 non-null     object         
 2   age                           91 non-null     object         
 3   medical_condition             91 non-null     object         
 4   admission_date                87 non-null     object         
 5   discharge_date                91 non-null     object         
 6   has_insurance_verify          91 non-null     object         
 7   medical_condition_clean       91 non-null     category       
 8   has_insurance_verify_clean    91 non-null     category       
 9   age_clean                     91 non-null     float64        
 10  admission_date_clean          87 non-null     datetime64[ns] 
 11  admission_date_text       

In [46]:
column_dic= {
    "admission_day_names": "admission_day_names", 
    "admission_month_name": "admission_month_name", 
    "admission_month": "admission_month", 
    "admission_day_of_month": "admission_day_of_month", 
    "admission_day_of_week": "admission_day_of_week", 
    "admission_year": "admission_year", 
    "year_after_admission_date": "year_after_admission_date", 
    "a_week_after_admission": "a_week_after_admission", 
    "a_week_after_admission_m2": "a_week_after_admission_m2", 
}

the_series= pd.Series(index=df1.columns, data= df1.columns)

print(the_series)

map_column_inside_series= (the_series.map(column_dic)
                           .dropna())

print(map_column_inside_series)

pick_columns_of_interest= map_column_inside_series.index

print(pick_columns_of_interest)

#By now we have only our columns of interest

ready_df= df1[pick_columns_of_interest].copy()

print(ready_df.head())

print(ready_df.columns)

path_ready_df= (r"D:\Public health\my_ready_df.csv")
ready_df.to_csv(path_ready_df, index=False, encoding= "utf-8-sig")
print("File written successfully")




    

patient_id                                        patient_id
patient_name                                    patient_name
age                                                      age
medical_condition                          medical_condition
admission_date                                admission_date
discharge_date                                discharge_date
has_insurance_verify                    has_insurance_verify
medical_condition_clean              medical_condition_clean
has_insurance_verify_clean        has_insurance_verify_clean
age_clean                                          age_clean
admission_date_clean                    admission_date_clean
admission_date_text                      admission_date_text
discharge_date_clean                    discharge_date_clean
discharge_date_text                      discharge_date_text
length_of_stay_days                      length_of_stay_days
length_of_stay                                length_of_stay
Onset_date              